In [2]:
"""
    test_arccos_signedmag.py

    Standalone test module for the signed-magnitude quantum arccos circuit implemented in
    the `quantum.quantum_cordic` module.

    This script:
      - imports `build_arccos_signedmag_circuit`
      - prepares a computational-basis signed-magnitude input |mag, sgn⟩
      - runs the circuit on AerSimulator (method="matrix_product_state")
      - measures thetaReg into a classical register
      - compares thetaReg / 2^p to numpy.arccos(x), or optionally numpy.arccos(x_loaded)

    Notes:
      - The `_init_path` import is retained to match the expected local environment.
      - Ensure `quantum/quantum_cordic.py` is discoverable on PYTHONPATH (same directory is fine).
"""

import _init_path

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np

# --- qiskit / aer ---
from qiskit import QuantumCircuit, ClassicalRegister, transpile
from qiskit_aer import AerSimulator

# --- import the circuit builder from your module ---
from quantum.quantum_cordic import build_arccos_signedmag_circuit


# =============================================================================
# Helpers
# =============================================================================

def quantize_signed_mag(x: float, m: int) -> Tuple[int, int, float]:
    """
    Quantize a real x into signed-magnitude fixed-point form.

    The representation is:
      - sign = 0 if x ≥ 0 else 1
      - u = round(|x| · 2^(m-1)), clipped to [0, 2^(m-1) - 1]
      - x_loaded = (-1)^sign · u / 2^(m-1)

    Args:
        x: Real value to encode, required to satisfy x ∈ [-1, 1).
        m: Number of qubits in the magnitude register; scale is 2^(m-1).

    Returns:
        A tuple (sign, u, x_loaded) where:
          - sign is the sign bit (0 for nonnegative, 1 for negative),
          - u is the integer magnitude loaded into magReg,
          - x_loaded is the exact real value represented by (sign, u).

    Raises:
        ValueError: If x is not in [-1, 1).
    """
    if not (-1.0 <= x < 1.0):
        raise ValueError("x must be in [-1, 1).")

    sign = 1 if x < 0 else 0
    scale = 2 ** (m - 1)
    u = int(round(abs(x) * scale))
    u = min(u, scale - 1)
    x_loaded = (-u / scale) if sign else (u / scale)
    return sign, u, x_loaded


def prepend_signed_mag_prep(core: QuantumCircuit, x: float, m: int) -> QuantumCircuit:
    """
    Prepend computational-basis state preparation for a signed-magnitude input.

    This routine constructs a new circuit PREP ∘ core, where PREP initializes the
    existing mag and sgn registers of `core` to encode x in signed-magnitude format.

    Implementation details:
      - Loads the magnitude integer u into magReg (little-endian: magReg[0] is LSB).
      - Applies X to the sign qubit if x < 0.

    Args:
        core: A circuit produced by `build_arccos_signedmag_circuit` containing registers
            named "mag" and "sgn".
        x: Real value to prepare in signed-magnitude format.
        m: Number of qubits in the magnitude register.

    Returns:
        A new QuantumCircuit with preparation gates prepended to `core`.

    Raises:
        StopIteration: If `core` does not contain registers named "mag" and "sgn".
        ValueError: If x is not in [-1, 1).
    """
    magReg = next(qr for qr in core.qregs if qr.name == "mag")
    signQ = next(qr for qr in core.qregs if qr.name == "sgn")

    sign, u, _ = quantize_signed_mag(x, m)

    prep = QuantumCircuit(*core.qregs)
    # magnitude u into magReg (little-endian: magReg[0] is LSB)
    for i in range(len(magReg)):
        if (u >> i) & 1:
            prep.x(magReg[i])
    if sign:
        prep.x(signQ[0])

    return prep.compose(core)


def measure_theta_only(circ: QuantumCircuit) -> Tuple[QuantumCircuit, int]:
    """
    Add a classical register and measure only thetaReg.

    This appends a new ClassicalRegister named "ctheta" and measures the quantum
    register named "theta" into it.

    Args:
        circ: Circuit containing a quantum register named "theta".

    Returns:
        A tuple (circ, w) where w is the width of thetaReg.

    Raises:
        StopIteration: If `circ` does not contain a register named "theta".
    """
    thetaReg = next(qr for qr in circ.qregs if qr.name == "theta")
    w = len(thetaReg)
    ctheta = ClassicalRegister(w, "ctheta")
    circ.add_register(ctheta)
    circ.measure(thetaReg, ctheta)
    return circ, w


def extract_theta_from_counts(
    counts: Dict[str, int],
    w: int,
    reverse_bits: bool = False
) -> Tuple[str, int, float]:
    """
    Extract the most likely measured theta value from simulator counts.

    Because the classical register for theta (ctheta) is added last, its bits
    appear as the leftmost chunk in the count key (after removing spaces).

    Args:
        counts: Measurement outcome counts from Qiskit Aer.
        w: Number of bits in thetaReg / ctheta.
        reverse_bits: If True, interpret theta bits as little-endian by reversing
            the extracted bitstring before converting to an integer.

    Returns:
        A tuple (theta_bits, theta_int, best_frac) where:
          - theta_bits is the most likely theta bitstring (length w),
          - theta_int is the corresponding integer value,
          - best_frac is the fraction of shots achieving the most likely outcome.

    Raises:
        ValueError: If counts is empty.
    """
    if not counts:
        raise ValueError("counts is empty.")

    best_key = max(counts, key=counts.get).replace(" ", "")
    theta_bits = best_key[:w]
    theta_int = int(theta_bits[::-1], 2) if reverse_bits else int(theta_bits, 2)
    best_frac = counts[max(counts, key=counts.get)] / sum(counts.values())
    return theta_bits, theta_int, best_frac


@dataclass
class TestResult:
    """
    Container for a single arccos test outcome.

    Fields:
        x: User-provided real input value.
        x_loaded: Quantized value actually prepared on the magnitude+sign registers.
        measured_theta_int: Most likely measured integer in thetaReg.
        measured_theta_angle: Measured angle decoded as measured_theta_int / 2^p.
        true_angle: Reference value computed via numpy.arccos on the chosen truth input.
        error: Absolute error |measured_theta_angle - true_angle|.
        best_fraction: Fraction of shots yielding the most likely theta outcome.
        theta_bits: Bitstring corresponding to the most likely measured theta value.
    """
    x: float
    x_loaded: float
    measured_theta_int: int
    measured_theta_angle: float
    true_angle: float
    error: float
    best_fraction: float
    theta_bits: str


# =============================================================================
# Test runner
# =============================================================================

def run_one(
    x: float,
    m: int,
    p: int,
    shots: int = 2048,
    clean: bool = True,
    reverse_bits: bool = False,
    compare_to_loaded_x: bool = False,
) -> TestResult:
    """
    Execute one end-to-end simulation test of the signed-magnitude arccos circuit.

    Workflow:
      1) Build the arccos core circuit with parameters (m, p).
      2) Prepend basis-state preparation for the signed-magnitude encoding of x.
      3) Measure thetaReg only.
      4) Simulate using AerSimulator(method="matrix_product_state").
      5) Decode thetaReg as theta_int / 2^p and compare to a classical reference.

    Args:
        x: Real input value in [-1, 1) to be encoded and tested.
        m: Number of magnitude qubits (magReg width).
        p: Number of fractional bits used to decode thetaReg.
        shots: Number of simulator shots.
        clean: If True, request CORDIC ancilla cleanup in the core circuit.
        reverse_bits: If True, reverse extracted theta bits before integer conversion.
        compare_to_loaded_x: If False, compare against arccos(x). If True, compare
            against arccos(x_loaded), where x_loaded is the quantized prepared value.

    Returns:
        A TestResult containing measured and reference angles, error, and metadata.

    Raises:
        ValueError: If x is not in [-1, 1) or if simulator counts are empty.
        StopIteration: If required registers ("mag", "sgn", "theta") are missing.
    """
    core = build_arccos_signedmag_circuit(m=m, p=p, clean_cordic_ancillas=clean)
    circ = prepend_signed_mag_prep(core, x=x, m=m)
    circ, w = measure_theta_only(circ)

    backend = AerSimulator(method="matrix_product_state")
    tqc = transpile(circ, backend=backend, optimization_level=0)
    result = backend.run(tqc, shots=shots).result()
    counts = result.get_counts()

    theta_bits, theta_int, best_frac = extract_theta_from_counts(counts, w=w, reverse_bits=reverse_bits)
    theta_angle = theta_int / (2 ** p)

    _, _, x_loaded = quantize_signed_mag(x, m)
    truth_x = x_loaded if compare_to_loaded_x else x
    true_angle = float(np.arccos(truth_x))
    err = abs(theta_angle - true_angle)

    return TestResult(
        x=x,
        x_loaded=x_loaded,
        measured_theta_int=theta_int,
        measured_theta_angle=theta_angle,
        true_angle=true_angle,
        error=err,
        best_fraction=best_frac,
        theta_bits=theta_bits,
    )


def run_suite(
    xs: List[float],
    m: int,
    p: int,
    shots: int = 2048,
    clean: bool = True,
    reverse_bits: bool = False,
    compare_to_loaded_x: bool = False,
) -> List[TestResult]:
    """
    Run a batch of tests over a list of input values.

    Args:
        xs: List of real inputs to test (each must satisfy x ∈ [-1, 1)).
        m: Number of magnitude qubits (magReg width).
        p: Number of fractional bits used to decode thetaReg.
        shots: Number of simulator shots per input.
        clean: If True, request CORDIC ancilla cleanup in the core circuit.
        reverse_bits: If True, reverse extracted theta bits before integer conversion.
        compare_to_loaded_x: If False, compare against arccos(x). If True, compare
            against arccos(x_loaded) for each prepared input.

    Returns:
        A list of TestResult objects in the same order as xs.
    """
    return [
        run_one(
            x=x,
            m=m,
            p=p,
            shots=shots,
            clean=clean,
            reverse_bits=reverse_bits,
            compare_to_loaded_x=compare_to_loaded_x,
        )
        for x in xs
    ]


def main() -> None:
    """
    Execute a simple default test suite and print results.

    Edit the parameters in this function to change the tested bit-widths,
    shot count, and input values.
    """
    # Edit parameters here
    m = 5
    p = 5
    shots = 2048
    xs = [0.0, 0.25, -0.25, 0.7, -0.7]

    results = run_suite(xs, m=m, p=p, shots=shots, clean=True, reverse_bits=False, compare_to_loaded_x=False)

    for r in results:
        print(
            f"x={r.x:+.3f} (loaded={r.x_loaded:+.6f})  "
            f"meas={r.measured_theta_angle:.6f}  true={r.true_angle:.6f}  "
            f"|err|={r.error:.3e}  best={r.best_fraction:.3f}"
        )


if __name__ == "__main__":
    main()

x=+0.000 (loaded=+0.000000)  meas=1.562500  true=1.570796  |err|=8.296e-03  best=1.000
x=+0.250 (loaded=+0.250000)  meas=1.312500  true=1.318116  |err|=5.616e-03  best=1.000
x=-0.250 (loaded=-0.250000)  meas=1.812500  true=1.823477  |err|=1.098e-02  best=1.000
x=+0.700 (loaded=+0.687500)  meas=0.812500  true=0.795399  |err|=1.710e-02  best=1.000
x=-0.700 (loaded=-0.687500)  meas=2.312500  true=2.346194  |err|=3.369e-02  best=1.000
